# Промпт #18 — Temperature edge case (JSON)

**Техника:** Temperature  
**Задача:** Проверить влияние температуры на структурированный вывод (JSON)  
**Сложность:** ⭐⭐⭐☆☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion
import json

In [2]:
prompt = """Извлеки данные из текста и верни строго в JSON формате.
Текст: Илон Маск, 52 года, живёт в Техасе, CEO компании Tesla.
JSON должен содержать поля: имя, возраст, город, должность, компания.
Верни только JSON, без пояснений."""

for temp in [0.1, 0.9]:
    print(f"=== ТЕМПЕРАТУРА {temp} ===")
    response = get_completion(prompt, temperature=temp)
    print(response)
    try:
        parsed = json.loads(response)
        print("✅ Валидный JSON")
    except json.JSONDecodeError:
        print("❌ Невалидный JSON")
    print()

=== ТЕМПЕРАТУРА 0.1 ===
```
{
  "имя": "Илон Маск",
  "возраст": 52,
  "город": "Техас",
  "должность": "CEO",
  "компания": "Tesla"
}
```
❌ Невалидный JSON

=== ТЕМПЕРАТУРА 0.9 ===
```json
{
    "имя": "Илон Маск",
    "возраст": 52,
    "город": "Техас",
    "должность": "CEO",
    "компания": "Tesla"
}
```
❌ Невалидный JSON



In [3]:
prompt = """Извлеки данные из текста и верни строго в JSON формате.
Текст: Илон Маск, 52 года, живёт в Техасе, CEO компании Tesla.
JSON должен содержать поля: имя, возраст, город, должность, компания.
Верни только JSON, без пояснений."""

for temp in [0.1, 0.9]:
    print(f"=== ТЕМПЕРАТУРА {temp} ===")
    response = get_completion(prompt, temperature=temp)
    print(response)
    try:
        # убираем markdown-фенсы если модель их добавила
        clean = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        parsed = json.loads(clean)
        print("✅ Валидный JSON")
    except json.JSONDecodeError:
        print("❌ Невалидный JSON")
    print()

=== ТЕМПЕРАТУРА 0.1 ===
```
{
  "имя": "Илон Маск",
  "возраст": 52,
  "город": "Техас",
  "должность": "CEO",
  "компания": "Tesla"
}
```
✅ Валидный JSON

=== ТЕМПЕРАТУРА 0.9 ===
```
{
  "имя": "Илон Маск",
  "возраст": 52,
  "город": "Техас",
  "должность": "CEO",
  "компания": "Tesla"
}
```
✅ Валидный JSON



## Оценка: 5/5

## Инсайт
Обе температуры выдали валидный JSON с идентичной структурой. 
Температура не влияет на структурированный вывод когда задача чёткая.

Но был неожиданный баг: модель обернула JSON в markdown-фенсы (```json), 
хотя в промпте сказано "без пояснений". json.loads() это не парсит — 
пришлось добавить .removeprefix("```json") очистку.

Два вывода:
1. Для JSON-задач температура не критична — структура держится при любой.
2. Всегда добавляй очистку от markdown-фенсов при парсинге JSON — 
   модели игнорируют инструкцию "только JSON" и добавляют форматирование.
   Надёжнее: чисти программно, не надейся на промпт.